# Semana 19 · ¿Cómo cambio de mapa sin perder el cielo?

Este cuaderno acompaña a la sesión de proyecto de la semana 19: matrices, rotaciones y el paso de coordenadas ecuatoriales a eclípticas.

**Cómo se usa.** Cada recuadro gris es una celda de código: selecciónala y pulsa `Mayús + Intro` para ejecutarla. Ejecuta las celdas **en orden**, porque las de más abajo usan variables definidas en las de arriba. Puedes cambiar cualquier número y volver a ejecutar; no puedes estropear nada.

Tiempo estimado: 30-45 minutos.

## 1. Matrices con NumPy

Una matriz es un `np.array` de dos dimensiones: una lista de filas. El operador `@` multiplica matrices entre sí o una matriz por un vector. (El operador `*` multiplica elemento a elemento, que es otra cosa: cuidado con confundirlos.)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

A = np.array([[1, 2],
              [3, 4]])
v = np.array([1, -1])
print("A @ v =", A @ v)
print("Primera columna de A:", A @ np.array([1, 0]), "  segunda:", A @ np.array([0, 1]))

## 2. La matriz de rotación en el plano

$$R(\theta) = \begin{pmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{pmatrix}$$

La probamos girando una figura: una flecha en forma de casa, para que se vea bien la orientación.

In [ ]:
def R(theta_grados):
    t = np.radians(theta_grados)
    return np.array([[np.cos(t), -np.sin(t)],
                     [np.sin(t),  np.cos(t)]])

# Los vértices de una figura, como columnas de una matriz de 2 × N
casa = np.array([[0, 2, 2, 1, 0, 0],
                 [0, 0, 1, 2, 1, 0]], dtype=float)

plt.figure(figsize=(6, 6))
for ang in [0, 45, 90, 180, 270]:
    girada = R(ang) @ casa              # una sola multiplicación gira todos los vértices
    plt.plot(girada[0], girada[1], label=f"{ang}°")
plt.axhline(0, color="k", lw=0.5); plt.axvline(0, color="k", lw=0.5)
plt.gca().set_aspect("equal"); plt.legend(); plt.grid(alpha=0.3)
plt.title("Una figura girada con R(θ)")
plt.show()

**Ejercicio.** Comprueba con NumPy tres propiedades de las rotaciones:

1. $R(60^\circ)\,R(30^\circ) = R(90^\circ)$ (usa `np.allclose` para comparar matrices con decimales).
2. La inversa de $R(\theta)$ es su traspuesta: `R(40) @ R(40).T` debe dar la matriz identidad.
3. Una rotación no cambia la longitud de un vector.

In [ ]:
# Tu código aquí

In [ ]:
# Solución
print("1.", np.allclose(R(60) @ R(30), R(90)))
print("2.", np.round(R(40) @ R(40).T, 12))
w = np.array([3.0, -7.0])
print("3.", np.linalg.norm(w), np.linalg.norm(R(123) @ w))

## 3. De ecuatoriales a eclípticas

Pasos:

1. De $(\alpha, \delta)$ a un vector unitario: $(\cos\delta\cos\alpha,\ \cos\delta\sin\alpha,\ \sin\delta)$.
2. Girar los ejes un ángulo $\varepsilon = 23{,}44^\circ$ alrededor del eje $x$ con $R_x(\varepsilon)$.
3. Del vector nuevo, sacar la longitud $\lambda$ = `arctan2(y, x)` y la latitud $\beta = \arcsin z$.

Para el camino inverso, de eclípticas a ecuatoriales, se gira $-\varepsilon$.

In [ ]:
EPS = 23.44      # oblicuidad de la eclíptica, en grados (valor redondeado)

def Rx(angulo_grados):
    e = np.radians(angulo_grados)
    return np.array([[1, 0, 0],
                     [0,  np.cos(e), np.sin(e)],
                     [0, -np.sin(e), np.cos(e)]])

def a_vector(lon, lat):
    """Ángulos en grados → vector unitario."""
    lon, lat = np.radians(lon), np.radians(lat)
    return np.array([np.cos(lat) * np.cos(lon), np.cos(lat) * np.sin(lon), np.sin(lat)])

def a_angulos(v):
    """Vector unitario → (longitud entre 0 y 360°, latitud), en grados."""
    lon = np.degrees(np.arctan2(v[1], v[0])) % 360
    lat = np.degrees(np.arcsin(np.clip(v[2], -1, 1)))
    return lon, lat

def ecuatorial_a_ecliptica(alfa, delta):
    return a_angulos(Rx(EPS) @ a_vector(alfa, delta))

def ecliptica_a_ecuatorial(lam, beta):
    return a_angulos(Rx(-EPS) @ a_vector(lam, beta))

pruebas = {
    "punto Aries (equinoccio de marzo)": (0, 0),
    "solsticio de junio":               (90, 23.44),
    "equinoccio de septiembre":         (180, 0),
    "polo norte de la eclíptica":       (270, 66.56),
}
for nombre, (a, d) in pruebas.items():
    lam, bet = ecuatorial_a_ecliptica(a, d)
    print(f"{nombre:36s} α = {a:6.2f}°  δ = {d:+6.2f}°   →   λ = {lam:7.2f}°  β = {bet:+6.2f}°")

Todo sale como en la sesión 2. En el polo de la eclíptica, la longitud que aparece no significa nada: en un polo todas las longitudes coinciden, igual que en el Polo Norte de la Tierra todos los meridianos se juntan. El número que sale depende solo de redondeos.

**Ejercicio.** Régulo, la estrella más brillante de Leo, tiene aproximadamente $\alpha = 10^\text{h}\,08^\text{m}\,22^\text{s}$ y $\delta = +11^\circ\,58'$ (valores de memoria; compruébalos en Stellarium).

1. Pasa la ascensión recta a grados (cada hora son 15°).
2. Calcula sus coordenadas eclípticas. Deberías ver que está casi sobre la eclíptica: por eso la Luna y los planetas pasan a menudo cerca de ella.
3. Compara con lo que te da Stellarium y vuelve a las ecuatoriales con la función inversa.

In [ ]:
# Tu código aquí

In [ ]:
# Solución
alfa = 15 * (10 + 8/60 + 22/3600)
delta = 11 + 58/60
lam, bet = ecuatorial_a_ecliptica(alfa, delta)
print(f"Régulo: α = {alfa:.3f}°, δ = {delta:.3f}°  →  λ = {lam:.2f}°, β = {bet:+.2f}°")
print("De vuelta:", np.round(ecliptica_a_ecuatorial(lam, bet), 3))

Los valores que da Stellarium pueden diferir unas décimas de grado: Stellarium usa las coordenadas «de la fecha», y el eje de la Tierra se mueve muy despacio (la **precesión**, unos 50 segundos de arco al año), mientras que aquí usamos las de un catálogo con época 2000.

## 4. Un plano visto desde el otro

La eclíptica, dibujada en coordenadas ecuatoriales, no es una recta: es una onda que sube hasta $+23{,}44^\circ$ en $\alpha = 6^\text{h}$ y baja hasta $-23{,}44^\circ$ en $\alpha = 18^\text{h}$. Es la declinación del Sol a lo largo del año, la misma curva de la semana 1, ahora deducida con una rotación.

In [ ]:
lams = np.linspace(0, 360, 361)
ecl_en_ecuat = np.array([ecliptica_a_ecuatorial(l, 0) for l in lams])
ecu_en_ecl = np.array([ecuatorial_a_ecliptica(a, 0) for a in lams])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.8))
orden = np.argsort(ecl_en_ecuat[:, 0])
ax1.plot(ecl_en_ecuat[orden, 0] / 15, ecl_en_ecuat[orden, 1])
ax1.set_xlabel("ascensión recta (h)"); ax1.set_ylabel("declinación (°)")
ax1.set_title("La eclíptica en coordenadas ecuatoriales"); ax1.grid(alpha=0.3)
ax1.set_xlim(24, 0)      # en los mapas del cielo, la ascensión recta crece hacia la izquierda

orden = np.argsort(ecu_en_ecl[:, 0])
ax2.plot(ecu_en_ecl[orden, 0], ecu_en_ecl[orden, 1], color="tab:orange")
ax2.set_xlabel("longitud eclíptica (°)"); ax2.set_ylabel("latitud eclíptica (°)")
ax2.set_title("El ecuador celeste en coordenadas eclípticas"); ax2.grid(alpha=0.3)
ax2.set_xlim(360, 0)
plt.tight_layout(); plt.show()

## 5. Experimenta

1. El polo norte celeste está en $\delta = 90^\circ$. ¿Qué coordenadas eclípticas tiene?
2. Escribe la matriz $R_z(\theta)$, que gira alrededor del eje $z$, y comprueba que aplicarla a un vector equivale a sumar $\theta$ a su ascensión recta (con el signo que corresponda).
3. (Más difícil) Usa $R_z$ y $R_x$ para pasar de coordenadas ecuatoriales a horizontales (altura y azimut). Necesitas el **ángulo horario** de la estrella (la hora sidérea local menos su ascensión recta) y tu latitud.

In [ ]:
# Tu experimento aquí